# Путешествуем по сайту, собираем данные, сохраняем в файл

Код с прошлого занятия

In [ ]:
from bs4 import BeautifulSoup
import requests

def get_song_text(url):
    response = requests.get(url)

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')

        content_div = soup.select_one('body > main > div.container > div.content > article')

        if content_div:
            return content_div.get_text(separator='\n')
        else:
            raise ValueError(f'Не найден текст на странице {url}')
    else:
        raise IOError(f"Ошибка выполнения запроса: {response.status_code} {url}")

In [ ]:
get_song_text('https://korol-i-shut.su/songs/text/bednyazhka.html')

'Увидел я ее в смирительной рубашке,\nОна вздохнула: "Никому я не нужна,\nВчера соседу проиграла в шашки,\nКому пойдет такая глупая жена?"\nЯ утешал ее: "Не унывай, родная,\nСказать по правде, все мы не в своем уме,\nСвои пороки глубоко в себе скрывая,\nМы очень мало знаем о себе".\nПрипев:\nНет, я не в силах ей помочь,\nМне нелегко ее понять,\nМне проще удалиться прочь,\nЧем что-то нужное сказать,\nОт этих странных ее слов\nМеня колотит и трясет,\nОна в плену печальных снов,\nНе знаю, что ее спасет...\nЕе лицо на время прояснилось:\n"Я благодарна нашему врачу,\nОн добрый, я в него почти влюбилась\nИ от него ребеночка хочу..."\nЯ сделал вид, что рад ее желанью,\n"Чудесно!" - ей в ответ проговорил,\nЧто она чувствует, живя во тьме за гранью?\nДа, я ее по-прежнему любил.\n(Припев)  '

Но вот проблема. У КиШ больше 200 песен. Не все же url нам копировать вручную?

Чтобы не заниматься рутиной, давайте автоматизируем сбор ссылок на песни из одного альбома

### Сбор песен из одного альбома

Поймем, как из html получить все названия песен и url

In [ ]:
import requests
from bs4 import BeautifulSoup

response = requests.get('https://korol-i-shut.su/albums/kamnem-po-golove/')
print(response.status_code)

200


In [ ]:
bs = BeautifulSoup(response.content, 'html.parser')

In [ ]:
for a in bs.select('a.track-list__name'):
    print(a)
    print(a.get('href'))
    print(a.get_text())
    print('-----')

<a class="track-list__name" href="https://korol-i-shut.su/songs/text/smelchak-i-veter.html">Смельчак и Ветер</a>
https://korol-i-shut.su/songs/text/smelchak-i-veter.html
Смельчак и Ветер
-----
<a class="track-list__name" href="https://korol-i-shut.su/songs/text/prokaznik-skomoroh.html">Проказник Скоморох</a>
https://korol-i-shut.su/songs/text/prokaznik-skomoroh.html
Проказник Скоморох
-----
<a class="track-list__name" href="https://korol-i-shut.su/songs/text/vernaya-zhena.html">Верная Жена</a>
https://korol-i-shut.su/songs/text/vernaya-zhena.html
Верная Жена
-----
<a class="track-list__name" href="https://korol-i-shut.su/songs/text/sadovnik.html">Садовник</a>
https://korol-i-shut.su/songs/text/sadovnik.html
Садовник
-----
<a class="track-list__name" href="https://korol-i-shut.su/songs/text/bluzhdayut-teni.html">Блуждают Тени</a>
https://korol-i-shut.su/songs/text/bluzhdayut-teni.html
Блуждают Тени
-----
<a class="track-list__name" href="https://korol-i-shut.su/songs/text/vnezapnaya-gol

Соберем в функцию, которая возвращает словарь с названиями и ссылками

In [ ]:
class ResponseStatusError(Exception):
    pass


class ParsingError(Exception):
    pass

In [ ]:
def get_album_tracks(album_url):
    response = requests.get(album_url)

    if response.status_code != 200:
        raise ResponseStatusError(f'Статус != 200. Статус {response.status_code}, url: {album_url}')

    bs = BeautifulSoup(response.content, 'html.parser')

    a_tags = bs.select('a.track-list__name')

    if not a_tags:
        raise ParsingError(f'Не нашли ссылки на песни: {album_url}')

    tracks_info = [
        {
            'title': a.get_text().strip(),
            'url': a.get('href')
        }
        for a in a_tags
    ]

    return tracks_info

# url = 'https://korol-i-shut.su/albums/kamne'

# try:
#     get_album_tracks(url)
# except ResponseStatusError:
#     print(f'Ошибка с url: {url}')

# get_album_tracks('https://korol-i-shut.su/songs/text/smelchak-i-veter.html')
get_album_tracks('https://korol-i-shut.su/albums/kamnem-po-golove/')

[{'title': 'Смельчак и Ветер',
  'url': 'https://korol-i-shut.su/songs/text/smelchak-i-veter.html'},
 {'title': 'Проказник Скоморох',
  'url': 'https://korol-i-shut.su/songs/text/prokaznik-skomoroh.html'},
 {'title': 'Верная Жена',
  'url': 'https://korol-i-shut.su/songs/text/vernaya-zhena.html'},
 {'title': 'Садовник',
  'url': 'https://korol-i-shut.su/songs/text/sadovnik.html'},
 {'title': 'Блуждают Тени',
  'url': 'https://korol-i-shut.su/songs/text/bluzhdayut-teni.html'},
 {'title': 'Внезапная Голова',
  'url': 'https://korol-i-shut.su/songs/text/vnezapnaya-golova.html'},
 {'title': 'Шар Голубой',
  'url': 'https://korol-i-shut.su/songs/text/shar-goluboi.html'},
 {'title': 'Злодей и Шапка',
  'url': 'https://korol-i-shut.su/songs/text/zlodei-i-shapka.html'},
 {'title': 'От Женщин Кругом Голова',
  'url': 'https://korol-i-shut.su/songs/text/ot-zhenschin-krugom-golova.html'},
 {'title': 'Рыбак', 'url': 'https://korol-i-shut.su/songs/text/rybak.html'},
 {'title': 'Мотоцикл',
  'url': 

У КиШ 12 альбомов. Не хочется их все собирать руками. Давайте этот процесс тоже автоматизируем

### Сбор данных о альбомах

Поймем, как из html получить все названия альбомов и url

In [ ]:
get_album_tracks('https://korol-i-shut.su/albums/')

[{'title': 'Камнем по Голове',
  'url': 'https://korol-i-shut.su/albums/kamnem-po-golove/'},
 {'title': 'Король и Шут',
  'url': 'https://korol-i-shut.su/albums/korol-i-shut/'},
 {'title': 'Акустический альбом',
  'url': 'https://korol-i-shut.su/albums/akusticheskii-albom/'},
 {'title': 'Ели Мясо Мужики',
  'url': 'https://korol-i-shut.su/albums/eli-myaso-muzhiki/'},
 {'title': 'Герои и Злодеи',
  'url': 'https://korol-i-shut.su/albums/geroi-i-zlodei/'},
 {'title': 'Собрание', 'url': 'https://korol-i-shut.su/albums/sobranie/'},
 {'title': 'Как в старой сказке',
  'url': 'https://korol-i-shut.su/albums/kak-v-staroi-skazke/'},
 {'title': 'Жаль, нет ружья!',
  'url': 'https://korol-i-shut.su/albums/zhal-net-ruzhya/'},
 {'title': 'Мёртвый Анархист',
  'url': 'https://korol-i-shut.su/albums/mertvyi-anarhist/'},
 {'title': 'Бунт на корабле',
  'url': 'https://korol-i-shut.su/albums/bunt-na-korable/'},
 {'title': 'Я Алкоголик Анархист',
  'url': 'https://korol-i-shut.su/albums/ya-alkogolik-an

Соберем в функцию

Альбомы умеем искать, тексты тоже, осталось собрать все вместе

### Сбор данных о всех песнях

Давайте напишем функцию, котрая на вход будет получать ссылку на список альбомов, а возвращать список словарей со всеми композициями КиШ:
```
[
    {'title': 'Прыгнуть со скалы', 'url': 'https>//...'},
    {'title': 'Бедняжка', 'url': 'https>//...'}
]

```

In [ ]:
def get_all_tracks_info(albums_url):
    albums_info = get_album_tracks(albums_url)

    all_tracks_info = []
    for album_info in albums_info:
        album_url = album_info['url']
        album_tracks = get_album_tracks(album_url)
        all_tracks_info.extend(album_tracks)

    return all_tracks_info

all_tracks_info = get_all_tracks_info('https://korol-i-shut.su/albums/')
print(all_tracks_info)

[{'title': 'Смельчак и Ветер', 'url': 'https://korol-i-shut.su/songs/text/smelchak-i-veter.html'}, {'title': 'Проказник Скоморох', 'url': 'https://korol-i-shut.su/songs/text/prokaznik-skomoroh.html'}, {'title': 'Верная Жена', 'url': 'https://korol-i-shut.su/songs/text/vernaya-zhena.html'}, {'title': 'Садовник', 'url': 'https://korol-i-shut.su/songs/text/sadovnik.html'}, {'title': 'Блуждают Тени', 'url': 'https://korol-i-shut.su/songs/text/bluzhdayut-teni.html'}, {'title': 'Внезапная Голова', 'url': 'https://korol-i-shut.su/songs/text/vnezapnaya-golova.html'}, {'title': 'Шар Голубой', 'url': 'https://korol-i-shut.su/songs/text/shar-goluboi.html'}, {'title': 'Злодей и Шапка', 'url': 'https://korol-i-shut.su/songs/text/zlodei-i-shapka.html'}, {'title': 'От Женщин Кругом Голова', 'url': 'https://korol-i-shut.su/songs/text/ot-zhenschin-krugom-golova.html'}, {'title': 'Рыбак', 'url': 'https://korol-i-shut.su/songs/text/rybak.html'}, {'title': 'Мотоцикл', 'url': 'https://korol-i-shut.su/songs

In [ ]:
len(all_tracks_info)

297

Теперь давайте преобразуем ссылку на страницу в текст, чтобы получить:
```

[
    {'title': 'Прыгнуть со скалы', 'text': 'С головы сорвал'},
    {'title': 'Бедняжка', 'text': 'Увидел я ее в смирительной рубашке'}
]

```

In [ ]:
from time import sleep
all_tracks_text = []
for track in all_tracks_info[:10]:
    all_tracks_text.append( {'title': track['title'], 'text': get_song_text(track['url'])})
    sleep(0.5)

In [ ]:
all_tracks_text

[{'title': 'Смельчак и Ветер',
  'text': 'Припев:\nЯ ведь не из робких,\nВсе мне по плечу.\nСильный я и ловкий,\nВетра проучу!\nДул сильный ветер, крыши рвал.\nИ, несмотря на поздний час,\nВ округе вряд ли кто-то спал -\nСтихия не на шутку разошлась.\nНо вдруг какой-то парень с криком побежал\nИ принялся махать метлой:\n"Ах, ветер, негодяй, ты спать мне помешал,\nА ну-ка выходи на бой!"\n(Припев)\nИ ветер закружился, заметался\nИ ели начал с корнем рвать:\n"Откуда этот сумасшедший взялся,\nЧто хочет с ветром воевать".\nНо парень не сдавался и метлой махал,\nИ удалялся вглубь полей.\nИ впрямь неплохо с ветром воевал,\nА ветер становился злей...\n(Припев)\nНо вдруг метла со свистом улетела прочь\nИ храбрый парень вслед за ней.\nА после этого спокойней стала ночь -\nИсчез во мраке дуралей.\nЕго под утро пастухи нашли в стогу -\nОн очень крепко спал,\nА ветер песни напевал ему\nИ кудри ласково трепал.\n(Припев)  '},
 {'title': 'Проказник Скоморох',
  'text': 'На свадьбе скоморох,\nБыл прыт

### Сохраняем результата в файл

In [ ]:
import json

with open('texts.json', 'w', encoding='utf-8') as file:
    json.dump(all_tracks_text, file, ensure_ascii=False, indent=2)

## Домашнее задание

- Для вашего сайта сделайте сбор ссылок со страницы и информации по ссылкам, сохраните в формате JSON.
- Погуглите формат pickle, сохраните данные в нем.
- Опционально: Сохраните данные в формате csv, мы давно работали с ним по книге. Можете поискать информацию как в книге, так и в интернете



